# Ablation Study: SimMTM SSL Pre-Training

## Overview
This notebook implements the **Ablation Study** on the impact of the unlabeled pre-training datasets on downstream classification performance, for the **SimMTM** architecture.

The 5 available unlabeled datasets (Corponi Multisource) are:
| Kaggle folder | Real name |
|---|---|
| `dati_preelaborati` | BigIdeas Lab |
| `spd` | Stress Physiology Database (SPD) |
| `ue4w` | University E4 Wearable (UE4W) |
| `weee` | Wearable EEG/E4 dataset (WEEE) |
| `wesd` | Wearable Emotion/Stress Dataset (WESD) |

### Instructions
1. In the Kaggle **Data** panel, add the dataset containing all 5 unlabeled sources + `metadata.pkl`.
2. Run all cells. The loop iterates automatically over all 9 subsets.
3. Download all `.pt` files from `/kaggle/working/ablation_weights/`.

In [ ]:
# ── CELL 1: Setup & Dependencies ─────────────────────────────────────────────
import os, sys, shutil, glob, h5py, pickle
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
from einops import rearrange
from torch.utils.data import Dataset, DataLoader

WORK_DIR = '/kaggle/working/e4selflearning'
if not os.path.exists(WORK_DIR):
    print("Cloning repository...")
    !git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}

print("Installing dependencies...")
!pip install -q biosppy flirt ruamel.yaml hrv-analysis neurokit2 distinctipy joypy peakutils biopsykit einops

if WORK_DIR not in sys.path: sys.path.insert(0, f"{WORK_DIR}/src")
os.chdir(WORK_DIR)

def apply_patch(file_path, old_str, new_str):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f: c = f.read()
        if old_str in c:
            with open(file_path, 'w') as f: f.write(c.replace(old_str, new_str))
            print(f"Patch OK: {file_path}")

apply_patch('segment.py', 'newshape=', 'shape=')
apply_patch('src/timebase/data/preprocessing.py', 'newshape=', 'shape=')

static_p = 'src/timebase/data/static.py'
with open(static_p, 'r') as f: c = f.read()
if '"dati_preelaborati"' not in c:
    c = c.replace('    "wesd": 12,\n}', '    "wesd": 12,\n    "dati_preelaborati": 13,\n}')
    c = c.replace('    "wesd": "lawngreen",\n}', '    "wesd": "lawngreen",\n    "dati_preelaborati": "steelblue",\n}')
    with open(static_p, 'w') as f: f.write(c)
    print("Patch OK: static.py (dati_preelaborati)")

print("\n--- Setup complete ---")

In [ ]:
# ── CELL 2: SimMTM Architecture Definition ────────────────────────────────────
class PatchEmbedding(nn.Module):
    def __init__(self, patch_size, d_model):
        super().__init__()
        self.proj = nn.Linear(patch_size, d_model)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x): return self.norm(self.proj(x))

class SimMTMEncoder(nn.Module):
    def __init__(self, seg_len=640, patch_size=16, d_model=128, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        self.patch_size = patch_size
        self.d_model = d_model
        self.n_patches = seg_len // patch_size
        self.patch_emb = PatchEmbedding(patch_size, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, self.n_patches, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    def patchify(self, x):
        B, C, T = x.shape
        return rearrange(x[:, :, :self.n_patches * self.patch_size], 'b c (n p) -> (b c) n p', p=self.patch_size)

    def forward(self, x):
        patches = self.patchify(x)
        emb = self.patch_emb(patches) + self.pos_emb
        return self.norm(self.transformer(emb))

print("SimMTMEncoder defined.")

In [ ]:
# ── CELL 3: Ablation Configuration ────────────────────────────────────────────
ALL_DATASETS = ['dati_preelaborati', 'spd', 'ue4w', 'weee', 'wesd']

ABLATION_SUBSETS = [
    ('only_bigideas',            ['dati_preelaborati']),
    ('only_spd',                 ['spd']),
    ('only_ue4w',                ['ue4w']),
    ('only_weee',                ['weee']),
    ('only_wesd',                ['wesd']),
    ('bigideas_spd',             ['dati_preelaborati', 'spd']),
    ('bigideas_spd_ue4w',        ['dati_preelaborati', 'spd', 'ue4w']),
    ('bigideas_spd_ue4w_weee',   ['dati_preelaborati', 'spd', 'ue4w', 'weee']),
    ('all_datasets',             ALL_DATASETS),
]

print(f"Ablation study: {len(ABLATION_SUBSETS)} subsets to test.")
for tag, ds in ABLATION_SUBSETS:
    print(f"  [{tag}]: {ds}")

In [ ]:
# ── CELL 4: Helper functions ───────────────────────────────────────────────────
CHANNELS = ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']

class SSLDataset(Dataset):
    def __init__(self, h5_list):
        self.files = h5_list
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        with h5py.File(self.files[i], 'r') as f:
            sigs = []
            for c in CHANNELS:
                s = torch.tensor(f[c][:]).float() if c in f else torch.zeros(640)
                s = F.interpolate(s.view(1,1,-1), size=640, mode='linear').squeeze()
                s = (s - s.mean()) / (s.std() + 1e-8)
                sigs.append(s)
            return torch.stack(sigs)


def ingest_subset(datasets_to_use, src_dir, target_dir, seg_dir):
    """
    Symlinks only the selected dataset folders into target_dir,
    then remaps metadata.pkl using a filesystem walk (not key-prefix matching,
    because original metadata keys may have different path structures).
    """
    if os.path.exists(target_dir): shutil.rmtree(target_dir)
    os.makedirs(target_dir, exist_ok=True)

    # Copy metadata and symlink only selected dataset folders
    shutil.copy2(os.path.join(src_dir, 'metadata.pkl'), os.path.join(target_dir, 'metadata.pkl'))
    for ds in datasets_to_use:
        s_path = os.path.join(src_dir, ds)
        d_path = os.path.join(target_dir, ds)
        if os.path.exists(s_path):
            if not os.path.exists(d_path):
                os.symlink(s_path, d_path)
        else:
            print(f"  WARNING: dataset '{ds}' not found in {src_dir}")

    # REMAP metadata by walking the linked dirs.
    # The keys in metadata.pkl may NOT start with the folder name
    # (e.g. BigIdeas was originally uploaded with a different key).
    # Walking the disk finds exactly what is physically present.
    meta_p = os.path.join(target_dir, 'metadata.pkl')
    with open(meta_p, 'rb') as f: meta = pickle.load(f)
    new_sessions = {}
    for root, dirs, files in os.walk(target_dir, followlinks=True):
        if 'channels.h5' in files:
            clean_name = os.path.basename(root)
            new_rel_id = os.path.relpath(root, target_dir).replace('\\', '/')
            for old_id in meta['sessions_info'].keys():
                if old_id.endswith(clean_name):
                    new_sessions[new_rel_id] = meta['sessions_info'][old_id]
                    break
    meta['sessions_info'] = new_sessions
    with open(meta_p, 'wb') as f: pickle.dump(meta, f)
    print(f"  Sessions mapped via walk: {len(new_sessions)}")

    if len(new_sessions) == 0:
        print("  ERROR: 0 sessions mapped! Check that the dataset folders contain 'channels.h5' files.")
        return []

    # Segmentation
    ret = os.system(
        f"PYTHONPATH=src python segment.py "
        f"--data_dir {target_dir} "
        f"--output_dir {seg_dir} "
        f"--segment_length 640 --segmentation_mode 1 "
        f"--step_size 640 --num_workers 2 --overwrite"
    )
    if ret != 0:
        print(f"  WARNING: segmentation exited with code {ret}")
        return []

    seg_meta_p = os.path.join(seg_dir, 'metadata.pkl')
    if not os.path.exists(seg_meta_p):
        print("  WARNING: segmentation produced no metadata.pkl — skipping subset.")
        return []

    with open(seg_meta_p, 'rb') as f: seg_meta = pickle.load(f)
    h5_files = [os.path.join(WORK_DIR, p) for p in seg_meta['sessions_paths']]
    print(f"  Segments after segmentation: {len(h5_files)}")
    return h5_files


print("Helper functions defined.")

In [ ]:
# ── CELL 5: Main Ablation Loop ────────────────────────────────────────────────
import csv

meta_files = glob.glob('/kaggle/input/**/metadata.pkl', recursive=True)
if not meta_files:
    raise FileNotFoundError("metadata.pkl not found in /kaggle/input. Add the unlabeled dataset.")
SRC_DIR = os.path.dirname(meta_files[0])
print(f"Source directory: {SRC_DIR}")
print(f"Contents: {os.listdir(SRC_DIR)}")

TARGET_DIR = '/kaggle/working/e4selflearning/data/preprocessed/unsegmented_unlabelled'
SEG_DIR    = '/kaggle/working/e4selflearning/data/preprocessed/sl640_ss640_unlabelled'
OUTPUT_DIR = '/kaggle/working/ablation_weights'
os.makedirs(OUTPUT_DIR, exist_ok=True)

EPOCHS = 20
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

log_rows = []

for tag, datasets in ABLATION_SUBSETS:
    print(f"\n{'='*60}")
    print(f"SUBSET: [{tag}]  =>  {datasets}")
    print(f"{'='*60}")

    h5_files = ingest_subset(datasets, SRC_DIR, TARGET_DIR, SEG_DIR)
    if len(h5_files) == 0:
        print(f"  SKIP: no segments for [{tag}]")
        log_rows.append({'subset': tag, 'datasets': str(datasets), 'n_segments': 0, 'final_loss': 'N/A'})
        continue

    loader = DataLoader(SSLDataset(h5_files), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

    encoder = SimMTMEncoder().to(device)
    decoder = nn.Linear(128, 16).to(device)
    optimizer = torch.optim.AdamW(
        list(encoder.parameters()) + list(decoder.parameters()), lr=1e-4
    )

    final_loss = 0.0
    for epoch in range(1, EPOCHS + 1):
        encoder.train(); decoder.train()
        total_loss = 0.0
        for xb in loader:
            xb = xb.to(device)
            patches = encoder.patchify(xb)
            mask = torch.rand(*patches.shape[:2], device=device) < 0.4
            p_masked = patches.clone()
            p_masked[mask] = 0.0
            enc = encoder.norm(
                encoder.transformer(encoder.patch_emb(p_masked) + encoder.pos_emb)
            )
            loss = F.mse_loss(decoder(enc)[mask], patches[mask])
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / max(1, len(loader))
        final_loss = avg_loss
        if epoch % 5 == 0 or epoch == EPOCHS:
            print(f"  Epoch {epoch:2d}/{EPOCHS} | Loss: {avg_loss:.4f}")

    out_path = os.path.join(OUTPUT_DIR, f'simmtm_encoder_ablation_{tag}.pt')
    torch.save(encoder.state_dict(), out_path)
    print(f"  Saved: {out_path}")
    log_rows.append({'subset': tag, 'datasets': str(datasets), 'n_segments': len(h5_files), 'final_loss': f'{final_loss:.4f}'})

log_path = '/kaggle/working/ablation_pretraining_simmtm_log.csv'
with open(log_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['subset', 'datasets', 'n_segments', 'final_loss'])
    writer.writeheader(); writer.writerows(log_rows)

print(f"\n\n{'='*60}")
print("ALL SUBSETS COMPLETED.")
import pandas as pd
print(pd.DataFrame(log_rows).to_string(index=False))